In [44]:
# %%
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


try:
    movies_df = pd.read_csv('movies.csv')
except FileNotFoundError:
    print("Error: 'movies.csv' not found. Please upload your movie dataset.")
    exit() # Exit if the file is not found

# Handle missing genre data if necessary
movies_df['genres'] = movies_df['genres'].fillna('')

# Create a TF-IDF vectorizer for the genres
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_df['genres'])

# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def get_recommendations_by_partial_title(partial_title, year=None, cosine_sim=cosine_sim):
    """
    Generates content-based movie recommendations by searching for a partial title
    and optionally filtering by year.

    Args:
        partial_title (str): A part of the movie title to search for (e.g., 'Toy Story').
        year (int, optional): The release year to filter by. Defaults to None.
        cosine_sim (numpy.ndarray): The cosine similarity matrix.

    Returns:
        pandas.Series: A Series of recommended movie titles, or None if the movie is not found.
    """
    # Find movies in the DataFrame where the title contains the partial title (case-insensitive)
    matching_movies = movies_df[movies_df['title'].str.contains(partial_title, case=False, na=False)].copy()

    if matching_movies.empty:
        print(f"Error: No movies found containing '{partial_title}' in the title.")
        return None

    # If a year is specified, filter further
    if year is not None:
        # Note: This regex matches titles containing the year in parentheses,
        # which is a common format in datasets.
        matching_movies = matching_movies[matching_movies['title'].str.contains(f'\\({year}\\)', case=False, na=False)]
        if matching_movies.empty:
            print(f"Error: No movie found containing '{partial_title}' with year {year}.")
            return None

    # Assuming the first match is the desired movie if multiple match and no year is specified
    # Or the unique match if year was specified
    if len(matching_movies) > 1 and year is None:
        print(f"Multiple movies found for '{partial_title}'. Please be more specific or provide a year.")
        print("Matching titles:")
        print(matching_movies['title'])
        return None # Or you could choose the first one, depending on your preference

    # Get the index from the matching movie row
    # Use .iloc[0] to safely get the first row index from a potentially multi-row result before filtering
    idx = matching_movies.index[0]


    # Get the pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 5 most similar movies (excluding the movie itself)
    sim_scores = sim_scores[1:6]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 5 most similar movies
    return movies_df['title'].iloc[movie_indices]

# %%
# Example usage:
movie_title_part = 'Toy Story'
numbe_of_recommendations = 5 # Although the function returns 5, keeping this for clarity

recommendations = get_recommendations_by_partial_title(movie_title_part)

if recommendations is not None:
    print(f"Top {numbe_of_recommendations} recommendations for movies containing '{movie_title_part}':")

    # Create a list of dictionaries for the table data with S.No and Movie Title
    table_data = []
    for i, title in enumerate(recommendations):
        table_data.append({'S.No': i + 1, 'Movie Title': title}) # Only S.No and Movie Title

    # Create a pandas DataFrame from the list of dictionaries
    recommendations_df = pd.DataFrame(table_data)

    # Print the DataFrame as a table
    print(recommendations_df.to_string(index=False)) # Use to_string for better formatting in Colab output

    # Add the check here before attempting to iterate with iterrows
    # Print a header for the formatted output
    print(f"{'S.No':<5} | {'Movie Title':<40}")
    print("-" * 5 + "-|-" + "-" * 40 + "-|-")

    # Iterating through the recommendations and print details

    for serial_number, movie_title in enumerate(recommendations, start=1):

     # Print the formatted movie details
        print(f"{serial_number:<5} | {movie_title:<40}")



Multiple movies found for 'Toy Story'. Please be more specific or provide a year.
Matching titles:
0         Toy Story (1995)
2496    Toy Story 2 (1999)
8599    Toy Story 3 (2010)
Name: title, dtype: object
